In [1]:
import re
from datetime import datetime, timedelta

from selenium.webdriver.common.by import By

from src.flights.models.models import Airport, SingleSearch
from src.flights.scrapers.scrapers import OneWayScraper

In [2]:
search_item = SingleSearch(
    origin=Airport("IST"),
    destination=Airport("ESB"),
    departure_date=datetime.today().date() + timedelta(days=10),
    direct_only=True,
)
scraper = OneWayScraper(search_item)
print(scraper.url)

elements = scraper.get_raw_flight_results()
elements

https://www.google.com/travel/flights?q=Flights%20to%20ESB%20Airport%20from%20IST%20on%202024-12-14%20oneway%20direct&curr=EUR&gl=IT
https://www.google.com/travel/flights/search?tfs=CBwQAhogEgoyMDI0LTEyLTE0KABqBwgBEgNJU1RyBwgBEgNFU0JAAUgBcAGCAQsI____________AZgBAg&tfu=EgoIABAAGAAgAigB&gl=IT&curr=EUR


[<selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.74")>,
 <selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.75")>]

In [22]:
e1 = elements[0]
e1

<selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.74")>

In [24]:
items = e1.find_elements(By.TAG_NAME, "li")
items

[<selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.76")>,
 <selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.77")>,
 <selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.78")>,
 <selenium.webdriver.remote.webelement.WebElement (session="6a03a8bf7a89bbd77f5c08decfaebecf", element="f.7E76FF0B855779099FDEFED938CE3097.d.B204B68B057FA7EA22FAE002F0A4E024.e.79")>]

In [25]:
items[0].text

'8:00\u202fAM\n – \n9:10\u202fAM\nTurkish Airlines\n1 hr 10 min\nIST–ESB\nNonstop\n56 kg CO2e\n-16% emissions\n€42'

In [26]:
# AIRLINE LOGO
data = items[0].find_element(By.CLASS_NAME, "EbY4Pc").get_attribute("style")
match = re.search(r"url\((.*?)\)", data)

airline_logo = match.group(1)
airline_logo

'https://www.gstatic.com/flights/airline_logos/70px/TK.png'

In [27]:
# DEPARTURE & ARRIVAL TIME
# TODO: handle +- X days and add date to the time (datetime object)

data = items[0].find_element(By.CLASS_NAME, "mv1WYe").text

dep_time, arr_time = data.replace("\n", "").replace("\u202f", "").split(" – ")

dep_time, arr_time

('8:00AM', '9:10AM')

In [28]:
# AIRLINE

airline = items[0].find_element(By.CSS_SELECTOR, ".sSHqwe.tPgKwe.ogfYpf").text
airline

'Turkish Airlines'

In [29]:
# FLIGHT TIME

flight_time = items[0].find_element(By.CSS_SELECTOR, ".gvkrdb.AdWm1c.tPgKwe.ogfYpf").text
flight_time

'1 hr 10 min'

In [30]:
# AIRPORT CODES

airport_codes = items[0].find_element(By.CSS_SELECTOR, ".PTuQse.sSHqwe.tPgKwe.ogfYpf").text
airport_codes = tuple(airport_codes.split("–"))
airport_codes

('IST', 'ESB')

In [31]:
# STOPS
stops = items[0].find_element(By.CLASS_NAME, "BbR8Ec").text
stops

'Nonstop'

In [32]:
from selenium.common.exceptions import NoSuchElementException

# ONLY HAND LUGGAGE
# TODO: test with a flight that has only hand luggage
try:
    only_hand_luggage = items[0].find_element(By.CSS_SELECTOR, ".vmWDCc.NMm5M") is not None
except NoSuchElementException:
    only_hand_luggage = False
only_hand_luggage

False

In [36]:
# PRICE
price = items[0].find_element(By.CLASS_NAME, "BVAVmf").text
price = int(price.replace("€", ""))
price

42